# 02 – Data Cleaning & Session Engineering

**Goal:** Transform `data/raw/events.csv` into:
1. `data/processed/clean.csv`: Leakage-safe early-session purchase prediction dataset.
2. `data/processed/customer_profiles.csv`: Customer-level behavioral profiles for K-Means segmentation.

All final repeatable logic lives in `ml/data/make_dataset.py` (`python -m ml.data.make_dataset`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from ml import config
from ml.data.make_dataset import clean_events, build_modeling_snapshots, build_customer_profiles, load_raw

## Step 1 – Clean Raw Events & Construct 30-Minute Sessions

In [ ]:
raw = load_raw()
events_clean, log, stats = clean_events(raw)
for line in log:
    print(" -", line)
events_clean.head()

## Step 2 – Build Leakage-Safe Early-Session Snapshots (`clean.csv`)

- **Observation window:** First 3 non-transaction events of each session.
- **Prediction target (`will_purchase`):** `Yes` if a transaction occurs *after* the 3rd event in that same session, `No` otherwise.
- **Leakage prevention:** Sessions where a transaction already occurred at or before event 3 are excluded.

In [ ]:
clean_df, sessions = build_modeling_snapshots(events_clean, max_rows=config.MAX_MODELING_ROWS)
print("Clean modeling dataset shape:", clean_df.shape)
clean_df.head()

## Step 3 – Build Customer Behavioral Profiles (`customer_profiles.csv`)

In [ ]:
cust_df = build_customer_profiles(events_clean, sessions, max_customers=config.MAX_CUSTOMER_PROFILES)
print("Customer profiles shape:", cust_df.shape)
cust_df.head()

## Cleaning Log Summary ✍️

| Issue / Step | Rows Affected | Action Taken | Methodological Reason |
|---|---|---|---|
| Exact duplicate rows | 460 | Removed (`drop_duplicates()`) | Prevent double-counting identical logged events |
| Unsorted timestamps | 2,755,641 | Sorted by `(visitorid, timestamp)` | Required for accurate inter-event gap calculation |
| Missing `session_id` | 1,761,675 sessions | Engineered with 30-min inactivity cutoff | Industry-standard e-commerce session boundary |
| Post-purchase target leakage | 1,574 sessions | Excluded sessions purchasing at/before event 3 | Ensure inputs only contain pre-purchase behavior |